# Benchmarking the Cleaned Data

We run **five data-processing operations** on `clean_data.csv` using four different libraries, and compare how fast each library completes the work. Every operation is run **5 times per library** and the average is reported.

| # | Operation                | What it does |
|---|--------------------------|--------------|
| 1 | **Category Summary**     | Group by **category × condition**; for each group compute listing count, the **mean / median / 25th / 75th / 90th percentile / min / max / standard deviation of price**, mean / max / total likes, unique sellers, and unique products (13 aggregations per group) |
| 2 | **Find Popular Listings**| Keep listings priced RM100–RM10,000 with at least 5 likes and condition brand new or like new |
| 3 | **Top 10 per Category**  | Pick the 10 most-liked listings inside every category |
| 4 | **Keyword Search**       | Find listings whose title mentions gaming, laptop, phone, SSD, or RAM |
| 5 | **Add Seller Stats**     | Compute each seller's average price / total listings / total likes and attach to every row |

**Libraries compared:** Pandas · Polars · Multiprocessing · PySpark

**Measurements (averaged over 5 runs):** Time (s) · CPU (%) · Peak Memory (MB) · Records Returned · Throughput (records/second)

---
## 0. Setup

In [1]:
import importlib
import os
import tempfile
import pandas as pd

import workloads, benchmark_utils
importlib.reload(workloads)
importlib.reload(benchmark_utils)
from workloads import WORKLOADS, PANDAS_ENGINES
from benchmark_utils import benchmark

CLEAN   = 'clean_data.csv'
RESULTS = 'workload_results.csv'
RUNS    = 'workload_runs.csv'
REPEATS = 5

# Friendly name -> internal key (only place technical keys appear)
OPERATIONS = {
    'Category Summary':      'agg_category',
    'Find Popular Listings': 'filter_hot',
    'Top 10 per Category':   'top_n',
    'Keyword Search':        'keyword_search',
    'Add Seller Stats':      'seller_enrich',
}
LIBRARIES = {
    'Pandas':          'pandas',
    'Polars':          'polars',
    'Multiprocessing': 'mp',
    'PySpark':         'spark',
}

averages: list[dict] = []
per_run:  list      = []
print(f'REPEATS = {REPEATS} — each library runs 5 times per operation, then averaged.')

REPEATS = 5 — each library runs 5 times per operation, then averaged.


---
## 1. Load the cleaned dataset

In [2]:
df_clean = pd.read_csv(CLEAN)
n_rows = len(df_clean)
print(f'rows: {n_rows:,}, cols: {df_clean.shape[1]}')
df_clean.head()

rows: 198,429, cols: 13


,listing_id,product,price,condition,seller,time_posted,likes,listing_url,source_url,category,buyer_protection,delivery_info,posted_at
0,1437686064,Gaming PC RTX 5060Ti + Ryzen 7 3700X | 16GB RAM,3000.0,lightly used,electroparts,2 hours ago,1,https://www.carousell.com.my/p/gaming-pc-rtx-5...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-05-21 12:46:22.579328
1,1437681886,PC Gaming Ryzen 5 5500 RTX 4060 1TB SSD,2550.0,like new,nadia_marissa,3 hours ago,3,https://www.carousell.com.my/p/pc-gaming-ryzen...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-05-21 11:46:22.579328
2,1431417654,HP All-in-One Desktop PC i3,500.0,lightly used,easy4you2016,1 month ago,27,https://www.carousell.com.my/p/hp-all-in-one-d...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-04-21 14:46:22.579328
3,1436800248,Gaming PC Ryzen 9 3900X,2400.0,lightly used,mr_raii,5 days ago,14,https://www.carousell.com.my/p/gaming-pc-ryzen...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-05-16 14:46:22.579328
4,1437391911,PC Gaming Ryzen 5 3600 RTX3060 (USED) - Forza ...,2000.0,lightly used,zulakmalhakim71,2 days ago,7,https://www.carousell.com.my/p/pc-gaming-ryzen...,https://www.carousell.com.my/categories/comput...,computers tech > desktops,False,No deliver info,2026-05-19 14:46:22.579328


---
## 2. Sanity check on a 5,000-row sample

Make sure all four libraries agree on how many records each operation returns before timing anything.

In [3]:
sample_df   = df_clean.head(5000).copy()
sample_path = os.path.join(tempfile.gettempdir(), 'clean_sample.csv')
sample_df.to_csv(sample_path, index=False)

def _rows(result):
    return result.count() if type(result).__module__.startswith('pyspark.') else len(result)

rows_seen = {}
for op_name, w_key in OPERATIONS.items():
    row_counts = {}
    for lib_name, e_key in LIBRARIES.items():
        fn  = WORKLOADS[w_key][e_key]
        arg = sample_df if e_key in PANDAS_ENGINES else sample_path
        row_counts[lib_name] = _rows(fn(arg))
    rows_seen[op_name] = row_counts

check_table = pd.DataFrame(rows_seen).T
check_table.index.name = 'Operation'
check_table

,Pandas,Polars,Multiprocessing,PySpark
Operation,,,,
Category Summary,25,25,25,25
Find Popular Listings,920,920,920,920
Top 10 per Category,40,40,40,40
Keyword Search,1714,1714,1714,1714
Add Seller Stats,5000,5000,5000,5000


---
## 3. Run benchmarks — one section per operation

Each cell below benchmarks one operation across all four libraries × 5 runs and shows two tables: averaged metrics and per-run detail. Cells are independent.

In [4]:
def run_operation(operation: str) -> None:
    """Benchmark one operation across all libraries × REPEATS runs."""
    w_key = OPERATIONS[operation]
    # Idempotent: drop any previous entries for this operation
    averages[:] = [m for m in averages if m['Operation'] != operation]
    per_run[:]  = [r for r in per_run  if r['Operation'].iloc[0] != operation]

    summary_rows, runs_for_op = [], []
    for lib_name, e_key in LIBRARIES.items():
        fn  = WORKLOADS[w_key][e_key]
        arg = df_clean if e_key in PANDAS_ENGINES else CLEAN
        m   = benchmark(fn, arg, repeats=REPEATS, n_rows=n_rows)

        averages.append({
            'Operation':           operation,
            'Library':             lib_name,
            'Time (s)':            m['wall_time_s'],
            'CPU (%)':             m['cpu_pct'],
            'Peak Memory (MB)':    m['peak_mem_mb'],
            'Process Memory (MB)': m['rss_mem_mb'],
            'Records Returned':    m['rows_out'],
            'Throughput':    m['throughput_rps'],
        })

        runs_df = m['runs_df'].rename(columns={
            'run': 'Run #',
            'wall_time_s': 'Time (s)',
            'cpu_pct': 'CPU (%)',
            'peak_mem_mb': 'Peak Memory (MB)',
            'rss_mem_mb': 'Process Memory (MB)',
            'rows_out': 'Records Returned',
            'throughput_rps': 'Throughput',
        })
        runs_df.insert(0, 'Operation', operation)
        runs_df.insert(1, 'Library', lib_name)
        per_run.append(runs_df)
        runs_for_op.append(runs_df)

        summary_rows.append({
            'Library':           lib_name,
            'Time (s)':          round(m['wall_time_s'], 3),
            'CPU (%)':           round(m['cpu_pct'], 1),
            'Peak Memory (MB)':  round(m['peak_mem_mb'], 1),
            'Throughput': int(m['throughput_rps']),
            'Records Returned': int(m['rows_out']),
        })

    print(f'\n== {operation} — average of {REPEATS} runs ==')
    display(pd.DataFrame(summary_rows))
    print(f'\n== {operation} — every individual run ==')
    detail = pd.concat(runs_for_op, ignore_index=True)
    display(detail[['Library', 'Run #', 'Time (s)', 'CPU (%)', 'Peak Memory (MB)', 'Throughput']].round(3))

### 3.1 Category Summary

In [5]:
run_operation('Category Summary')


== Category Summary — average of 5 runs ==


,Library,Time (s),CPU (%),Peak Memory (MB),Throughput,Records Returned
0,Pandas,5.368,99.6,17.1,36992,881
1,Polars,0.109,438.9,0.1,1891677,881
2,Multiprocessing,4.232,354.0,63.4,46888,881
3,PySpark,1.490,196.6,0.2,134045,881



== Category Summary — every individual run ==


,Library,Run #,Time (s),CPU (%),Peak Memory (MB),Throughput
0,Pandas,1,5.376,100.844,17.676,36906.843
1,Pandas,2,5.637,98.399,16.972,35200.376
2,Pandas,3,5.292,98.905,16.972,37493.618
3,Pandas,4,5.249,99.425,16.970,37803.815
4,Pandas,5,5.283,100.554,16.971,37558.231
5,Polars,1,0.127,441.336,0.060,1556868.797
6,Polars,2,0.141,353.777,0.064,1403994.006
7,Polars,3,0.096,454.808,0.057,2062789.453
8,Polars,4,0.091,396.673,0.063,2190233.252
9,Polars,5,0.088,547.895,0.064,2244504.370


### 3.2 Find Popular Listings

In [6]:
run_operation('Find Popular Listings')


== Find Popular Listings — average of 5 runs ==


,Library,Time (s),CPU (%),Peak Memory (MB),Throughput,Records Returned
0,Pandas,0.023,177.8,1.4,8674907,23276
1,Polars,0.083,491.0,0.1,2384423,23276
2,Multiprocessing,2.088,524.9,9.9,95107,23276
3,PySpark,1.168,156.8,0.1,170266,23276



== Find Popular Listings — every individual run ==


,Library,Run #,Time (s),CPU (%),Peak Memory (MB),Throughput
0,Pandas,1,0.023,137.892,1.451,8755752.847
1,Pandas,2,0.023,133.479,1.449,8475561.575
2,Pandas,3,0.022,212.553,1.450,8997701.020
3,Pandas,4,0.025,190.945,1.449,8082976.900
4,Pandas,5,0.022,214.085,1.450,9062547.104
5,Polars,1,0.083,565.160,0.057,2392408.589
6,Polars,2,0.084,484.384,0.057,2365928.455
7,Polars,3,0.081,367.526,0.057,2456515.872
8,Polars,4,0.084,540.825,0.058,2368340.017
9,Polars,5,0.085,497.273,0.057,2338924.794


### 3.3 Top 10 per Category

In [7]:
run_operation('Top 10 per Category')


== Top 10 per Category — average of 5 runs ==


,Library,Time (s),CPU (%),Peak Memory (MB),Throughput,Records Returned
0,Pandas,0.157,121.2,29.2,1274424,1363
1,Polars,0.131,467.3,0.1,1535717,1363
2,Multiprocessing,2.378,470.7,9.9,84201,1363
3,PySpark,1.423,168.1,0.1,140353,1363



== Top 10 per Category — every individual run ==


,Library,Run #,Time (s),CPU (%),Peak Memory (MB),Throughput
0,Pandas,1,0.150,125.149,29.168,1324441.383
1,Pandas,2,0.184,110.216,29.165,1076682.912
2,Pandas,3,0.134,116.461,29.164,1478986.774
3,Pandas,4,0.152,123.278,29.164,1304640.264
4,Pandas,5,0.167,130.897,29.165,1187373.553
5,Polars,1,0.114,452.070,0.057,1739710.991
6,Polars,2,0.127,515.732,0.058,1559410.026
7,Polars,3,0.161,428.303,0.097,1236185.281
8,Polars,4,0.131,453.704,0.058,1516261.129
9,Polars,5,0.122,486.846,0.058,1627019.333


### 3.4 Keyword Search

In [8]:
run_operation('Keyword Search')


== Keyword Search — average of 5 runs ==


,Library,Time (s),CPU (%),Peak Memory (MB),Throughput,Records Returned
0,Pandas,0.954,100.6,0.9,208541,18172
1,Polars,0.087,477.6,0.1,2278965,18172
2,Multiprocessing,2.203,530.8,9.9,90153,18172
3,PySpark,1.425,152.6,0.1,139492,18172



== Keyword Search — every individual run ==


,Library,Run #,Time (s),CPU (%),Peak Memory (MB),Throughput
0,Pandas,1,1.049,98.334,0.905,189211.288
1,Pandas,2,0.966,101.879,0.905,205365.911
2,Pandas,3,0.914,102.556,0.904,217067.596
3,Pandas,4,0.907,99.918,0.905,218777.350
4,Pandas,5,0.935,100.297,0.904,212285.535
5,Polars,1,0.087,613.239,0.058,2290529.169
6,Polars,2,0.089,402.773,0.058,2223914.571
7,Polars,3,0.087,485.950,0.058,2285666.565
8,Polars,4,0.086,419.677,0.058,2317248.483
9,Polars,5,0.087,466.273,0.058,2277468.523


### 3.5 Add Seller Stats

In [9]:
run_operation('Add Seller Stats')


== Add Seller Stats — average of 5 runs ==


,Library,Time (s),CPU (%),Peak Memory (MB),Throughput,Records Returned
0,Pandas,0.230,108.9,15.8,865769,198429
1,Polars,0.124,498.6,0.1,1599842,198429
2,Multiprocessing,3.384,409.8,100.2,58785,198429
3,PySpark,2.176,181.5,0.1,92886,198429



== Add Seller Stats — every individual run ==


,Library,Run #,Time (s),CPU (%),Peak Memory (MB),Throughput
0,Pandas,1,0.227,110.128,15.767,874104.218
1,Pandas,2,0.211,111.170,15.765,941200.187
2,Pandas,3,0.232,107.863,15.766,856128.079
3,Pandas,4,0.245,108.378,15.766,809616.215
4,Pandas,5,0.234,106.814,15.769,847799.504
5,Polars,1,0.121,464.501,0.059,1638587.387
6,Polars,2,0.121,479.292,0.059,1645065.897
7,Polars,3,0.124,556.114,0.059,1605078.879
8,Polars,4,0.129,437.344,0.060,1542787.434
9,Polars,5,0.127,555.505,0.059,1567691.627


---
## 4. Save results to disk

Two CSVs with friendly column names: `workload_results.csv` (20 averaged rows) and `workload_runs.csv` (100 per-run rows).

In [10]:
avg_cols = ['Operation', 'Library', 'Time (s)', 'CPU (%)', 'Peak Memory (MB)', 'Process Memory (MB)', 'Records Returned', 'Throughput']
run_cols = ['Operation', 'Library', 'Run #', 'Time (s)', 'CPU (%)', 'Peak Memory (MB)', 'Process Memory (MB)', 'Records Returned', 'Throughput']

results = pd.DataFrame(averages)[avg_cols]
results.to_csv(RESULTS, index=False)

runs = pd.concat(per_run, ignore_index=True)[run_cols]
runs.to_csv(RUNS, index=False)

print(f'wrote {len(results)} averaged rows -> {RESULTS}')
print(f'wrote {len(runs)} per-run rows  -> {RUNS}')

wrote 20 averaged rows -> workload_results.csv
wrote 100 per-run rows  -> workload_runs.csv


### Averaged results

In [11]:
results.round(3)

,Operation,Library,Time (s),CPU (%),Peak Memory (MB),Process Memory (MB),Records Returned,Throughput
0,Category Summary,Pandas,5.368,99.625,17.112,266.270,881.0,36992.577
1,Category Summary,Polars,0.109,438.898,0.062,458.840,881.0,1891677.975
2,Category Summary,Multiprocessing,4.232,353.962,63.441,505.210,881.0,46888.453
3,Category Summary,PySpark,1.490,196.560,0.198,507.047,881.0,134045.927
4,Find Popular Listings,Pandas,0.023,177.791,1.450,508.211,23276.0,8674907.889
5,Find Popular Listings,Polars,0.083,491.034,0.057,488.026,23276.0,2384423.546
6,Find Popular Listings,Multiprocessing,2.088,524.919,9.936,476.845,23276.0,95107.196
7,Find Popular Listings,PySpark,1.168,156.783,0.112,476.112,23276.0,170266.937
8,Top 10 per Category,Pandas,0.157,121.200,29.165,477.329,1363.0,1274424.977
9,Top 10 per Category,Polars,0.131,467.331,0.066,601.076,1363.0,1535717.352


### Every individual run

In [12]:
runs.round(3)

,Operation,Library,Run #,Time (s),CPU (%),Peak Memory (MB),Process Memory (MB),Records Returned,Throughput
0,Category Summary,Pandas,1,5.376,100.844,17.676,266.246,881,36906.843
1,Category Summary,Pandas,2,5.637,98.399,16.972,266.293,881,35200.376
2,Category Summary,Pandas,3,5.292,98.905,16.972,266.297,881,37493.618
3,Category Summary,Pandas,4,5.249,99.425,16.970,266.250,881,37803.815
4,Category Summary,Pandas,5,5.283,100.554,16.971,266.262,881,37558.231
...,...,...,...,...,...,...,...,...,...
95,Add Seller Stats,PySpark,1,2.800,230.438,0.100,773.922,198429,70858.046
96,Add Seller Stats,PySpark,2,2.097,186.996,0.100,773.922,198429,94611.233
97,Add Seller Stats,PySpark,3,2.063,186.299,0.098,772.922,198429,96174.501
98,Add Seller Stats,PySpark,4,2.025,166.647,0.100,772.922,198429,97977.930


---
## 5. Next step

Open **`evaluation_charts.ipynb`** for the comparison charts, speedup heatmap, and per-run variance plot.